# Module 6 · One question, four approaches

"How did Acme's Q3 margin compare to Q2, and why?" through every version we built:

| Version | What it is |
|---|---|
| Module 1 | the whole filing in every prompt (`module_1_just_upload_the_pdf.ipynb`) |
| Module 2 | basic RAG, top 5 chunks (`module_2_send_only_what_matters.ipynb`) |
| Module 3 | a LangGraph agent with `search_docs` (`module_3_give_the_model_judgment.ipynb`) |
| Module 4 | agentic RAG, served on AgentShip (`module_4_search_with_judgment.ipynb`, `agents/agentic_rag.yaml`) |

Each version runs `RUNS` times, so p95 latency means something. With a self-hosted model, Module 1's
first run reads the whole ~100K-token filing before it writes a word; run this notebook once
before the lecture to warm the server's cache.

**Setup:** see `README.md` (`pip install -e .`, then `.env` names the model and server). Run with the *Agentic RAG demos* kernel. Module 4 needs Postgres for its checkpoints: `docker compose up -d`.

In [1]:
import os
import sys

os.environ.setdefault("AGENTSHIP_OTEL_EXPORTERS", "") # set to phoenix to see v3's traces at :6006

import numpy as np
import pandas as pd

from tools.settings import ROOT

sys.path.insert(0, str(ROOT)) # eval.py and agents/ live at the repo root, one level up
import eval as E # the same version functions eval.py scores

QUESTION = "How did Acme's Q3 margin compare to Q2, and why?"
VERSIONS = ["v0", "v1", "v2", "v3"]
RUNS = 3

In [2]:
rows, answers = [], {}
for version in VERSIONS:
    for i in range(RUNS):
        run = E.VERSIONS[version](QUESTION)
        rows.append({"version": version, "input_tok": run.prompt_tokens, "searches": len(run.searches),
                     "cost": run.cost, "latency": run.latency})
        print(f"{version} run {i + 1}: {run.prompt_tokens:,} input tokens, {len(run.searches)} searches, "
              f"{run.latency:.1f}s")
    answers[version] = run

v0 run 1: 106,842 input tokens, 0 searches, 2.9s


v0 run 2: 106,842 input tokens, 0 searches, 2.0s


v0 run 3: 106,842 input tokens, 0 searches, 3.3s


v1 run 1: 2,737 input tokens, 1 searches, 1.6s


v1 run 2: 2,737 input tokens, 1 searches, 1.6s


v1 run 3: 2,737 input tokens, 1 searches, 1.6s


... (2 outputs left out)

v2 run 3: 5,582 input tokens, 2 searches, 2.2s


v3 run 1: 21,064 input tokens, 5 searches, 17.6s


v3 run 2: 18,407 input tokens, 4 searches, 10.6s


v3 run 3: 8,813 input tokens, 2 searches, 4.8s


## The answers

In [3]:
for version, run in answers.items():
    searched = f" searched: {run.searches}" if run.searches else ""
    print(f"── {version} ──{searched}\n{run.answer}\n")

── v0 ──
Acme’s operating margin improved in Q3 2026 compared to Q2 2026:

* **Q3 2026 Operating Margin:** **21.0%** (Operating income of $885M on $4,212M revenue)
* **Q2 2026 Operating Margin:** **19.4%** (Operating income of $759M on $3,911M revenue)

**Why:**
According to the MD&A section, the margin expansion was driven primarily by **lower data-center costs per unit of compute** (as newer facilities with lower power costs came online and server utilization improved) combined with **operating leverage** on higher Cloud Services revenue growth.

── v1 ── searched: ["How did Acme's Q3 margin compare to Q2, and why?"]
Based on the provided context, there is no mention of Acme's financial results or operating margin for **Q2**. The context only compares Q3 2026 results (including gross and operating margins) to Q3 2025 and the first nine months (9M) of 2026 and 2025. 

Therefore, the provided context does not contain the answer.

── v2 ── searched: ['Q2 2026 gross margin operating marg

## The numbers, side by side

In [4]:
df = pd.DataFrame(rows)
table = df.groupby("version", sort=False).agg(
    input_tok=("input_tok", "median"), searches=("searches", "median"),
    usd_per_query=("cost", "mean"), p95_s=("latency", lambda s: np.percentile(s, 95)))
table["input_tok"] = table["input_tok"].map(lambda n: f"{n / 1000:.1f}K")
table["usd_per_query"] = table["usd_per_query"].map(lambda c: f"${c:.3f}" if pd.notna(c) else "n/a")
table["p95_s"] = table["p95_s"].map(lambda s: f"{s:.1f}s")
table

,input_tok,searches,usd_per_query,p95_s
version,,,,
v0,106.8K,0.0,$0.161,3.2s
v1,2.7K,1.0,$0.005,1.6s
v2,5.6K,2.0,$0.010,5.0s
v3,18.4K,4.0,$0.027,16.9s


Same question, four versions. Module 4, served on AgentShip, is what ships.

$ is priced at `PRICE_AS` when the model has no list price (a self-hosted model). For the whole
golden set, with retrieval and answer quality scored too: `python eval.py --agents Module 1,Module 2,Module 3,Module 4`.